# Camada Silver — `deslocamento`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Padronização de Tipos (`id_aluno`, `data_referencia`, `distancia_km`, `tempo_minutos`, `custo_diario`), Limpeza Textual (`cidade_origem`, `regiao_origem`, `meio_transporte`), Detecção de Outliers (IQR e Z-Score) e Remoção de Duplicatas.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [2]:
df = load_bronze("deslocamento", BRONZE_DIR)
rows_in = len(df)

df["id_deslocamento"] = df["id_deslocamento"].astype(str).str.strip().astype("string")
df["id_aluno"] = normalize_id_aluno(df["id_aluno"])
df["data_referencia"] = parse_date_col(df["data_referencia"])

for col in ["cidade_origem", "regiao_origem", "meio_transporte"]:
    df[col], _ = fix_mojibake_text(df[col])
    df[col] = df[col].str.title().astype("category")

for col in ["distancia_km", "tempo_minutos", "custo_diario"]:
    df[col] = parse_br_float(df[col]).clip(lower=0)

df["flag_dist_outlier_iqr"], iqr_dist = detect_outliers_iqr(df["distancia_km"])
df["flag_custo_outlier_iqr"], iqr_custo = detect_outliers_iqr(df["custo_diario"])

dups_removed = int(df.duplicated(subset=["id_deslocamento"]).sum())
df = df.drop_duplicates(subset=["id_deslocamento"]).reset_index(drop=True)

log = {
    "dataset": "deslocamento",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "distancia_outliers_iqr": iqr_dist,
    "custo_outliers_iqr": iqr_custo,
}
save_silver_outputs(df, log, "deslocamento", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)

{
  "dataset": "deslocamento",
  "rows_in": 1000,
  "rows_out": 1000,
  "duplicates_removed": 0,
  "distancia_outliers_iqr": {
    "q1": 8.65,
    "q3": 27.1675,
    "iqr": 18.5175,
    "lower_bound": -19.1262,
    "upper_bound": 54.9437,
    "outliers": 3
  },
  "custo_outliers_iqr": {
    "q1": 5.0,
    "q3": 32.665,
    "iqr": 27.665,
    "lower_bound": -36.4975,
    "upper_bound": 74.1625,
    "outliers": 30
  }
}


,id_deslocamento,id_aluno,data_referencia,cidade_origem,regiao_origem,meio_transporte,distancia_km,tempo_minutos,custo_diario,flag_dist_outlier_iqr,flag_custo_outlier_iqr
0,DESK000001,25795,2026-05-13,Osasco,Grande Sp,Caminhada / Bicicleta,2.39,24.0,0.00,False,False
1,DESK000002,10860,2026-05-18,São Paulo,Capital - Zs,Ônibus / Metrô,28.37,86.0,12.50,False,False
2,DESK000003,86820,2026-03-01,Rio De Janeiro,Zona Sul Rj,Carro Próprio,30.14,99.0,42.38,False,False
3,DESK000004,64886,2026-01-12,Santo André,Abc,Carro Próprio,33.94,91.0,40.89,False,False
4,DESK000005,16265,2025-08-05,Campinas,Interior,Carro Próprio,38.73,103.0,52.11,False,False
